In [ ]:
import xarray as xr

# load data
ds = xr.open_mfdataset("../data/interim/*.nc")
ds = ds.rename({"valid_time": "time"})

t2m_monthly_thresholds = []
tp_monthly_thresholds = []

for month in range(1, 13):
    print(f"Processing month {month}:")

    ds_month = ds.sel(time=ds["time"].dt.month == month)
    ds_month.load()

    t2m_threshold = ds_month["t2m"].quantile(0.90, dim="time")
    tp_threshold = ds_month["tp"].quantile(0.95, dim="time")

    t2m_threshold = t2m_threshold.expand_dims(month=[month])
    tp_threshold = tp_threshold.expand_dims(month=[month])

    t2m_monthly_thresholds.append(t2m_threshold)
    tp_monthly_thresholds.append(tp_threshold)

print("Stitching 12 months back together...")
t2m_thresholds = xr.concat(t2m_monthly_thresholds, dim="month")
tp_thresholds = xr.concat(tp_monthly_thresholds, dim="month")

print("Generate extreme flags")
heat_extremes = ds["t2m"].groupby("time.month") > t2m_thresholds
rain_extremes = ds["tp"].groupby("time.month") > tp_thresholds

ds["concurrent_extreme"] = (heat_extremes & rain_extremes).astype(int)

output_path = "../data/processed/labelled_extremes_2000-2024.nc"
ds[["t2m", "tp", "u10", "v10", "d2m", "concurrent_extreme"]].to_netcdf(output_path)
print("Done!")

In [1]:
import xarray as xr

labelled_ds = xr.open_dataset("../data/processed/labelled_extremes_2000-2024.nc")

labelled_ds

<xarray.Dataset> Size: 4GB
Dimensions:             (time: 9132, latitude: 121, longitude: 121)
Coordinates:
  * time                (time) datetime64[ns] 73kB 2000-01-01 ... 2024-12-31
    month               (time) int64 73kB ...
  * latitude            (latitude) float64 968B 35.0 34.75 34.5 ... 5.5 5.25 5.0
  * longitude           (longitude) float64 968B 65.0 65.25 65.5 ... 94.75 95.0
Data variables:
    t2m                 (time, latitude, longitude) float32 535MB ...
    tp                  (time, latitude, longitude) float32 535MB ...
    u10                 (time, latitude, longitude) float32 535MB ...
    v10                 (time, latitude, longitude) float32 535MB ...
    d2m                 (time, latitude, longitude) float32 535MB ...
    concurrent_extreme  (time, latitude, longitude) int64 1GB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-22T11:53 GRIB to CDM+CF via cfgrib-0.9.1...

In [23]:
total_extremes = labelled_ds["concurrent_extreme"].sum().item()
total_extremes

total_data_points = labelled_ds["concurrent_extreme"].size
total_data_points

percent = (total_extremes / total_data_points) * 100
print(f"Extreme %: {percent:.3f}")

Extreme %: 0.160


In [ ]:
ds = xr.open_mfdataset("../data/interim/*.nc")
ds

<xarray.Dataset> Size: 3GB
Dimensions:     (valid_time: 9132, latitude: 121, longitude: 121)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 73kB 2000-01-01 ... 2024-12-31
  * latitude    (latitude) float64 968B 35.0 34.75 34.5 34.25 ... 5.5 5.25 5.0
  * longitude   (longitude) float64 968B 65.0 65.25 65.5 ... 94.5 94.75 95.0
Data variables:
    u10         (valid_time, latitude, longitude) float32 535MB dask.array<chunksize=(183, 61, 61), meta=np.ndarray>
    v10         (valid_time, latitude, longitude) float32 535MB dask.array<chunksize=(183, 61, 61), meta=np.ndarray>
    d2m         (valid_time, latitude, longitude) float32 535MB dask.array<chunksize=(183, 61, 61), meta=np.ndarray>
    t2m         (valid_time, latitude, longitude) float32 535MB dask.array<chunksize=(183, 61, 61), meta=np.ndarray>
    tp          (valid_time, latitude, longitude) float32 535MB dask.array<chunksize=(183, 61, 61), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-22T11:53 GRIB to CDM+CF via cfgrib-0.9.1...

In [ ]:
print()